# 01 — Generative AI & Prompting for CAs

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives

1. Distinguish AI / ML / Generative AI / LLM / GPT.
2. Understand tokens, context window, temperature.
3. Recognise **hallucinations** and why they happen.
4. Apply five prompting patterns useful to a CA:
   * role prompting
   * few-shot prompting
   * structured-output prompting
   * chain-of-thought *output* (we ask the model to **show** its reasoning)
   * grounded prompting (provide the source text in the prompt)


## Concepts in plain language

**AI** is the broad field. **Machine Learning** is the subset where the system learns from data. **Generative AI** is the kind of ML that creates new content — text, images, code. **LLMs** (Large Language Models) are the engines behind ChatGPT, Claude, Gemini. **GPT** (Generative Pre-trained Transformer) is one family of LLMs.

**Token** ≈ a piece of a word. "depreciation" is 1–3 tokens; "NPR 1,25,000" is several. **Context window** = how many tokens the model can read at once.

**Hallucination** = the model confidently produces something that *sounds right* but is wrong. Always verify before relying on AI output for professional work.

In [1]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


Project root: /Users/aayush/Documents/Kings/ICAN/ICANN/rag


In [2]:
from src.llm_client import ask_llm
# Helper for tidy printing
def show(title, text):
    print('=' * 70)
    print(title)
    print('-' * 70)
    print(text)
    print()

## Pattern 1 — Role prompting

We tell the model *who it is*. This shapes vocabulary and tone.

In [3]:
no_role = ask_llm('Explain the difference between provisions and contingent liabilities.')
with_role = ask_llm(
    'Explain the difference between provisions and contingent liabilities.',
    system='You are a senior Nepali Chartered Accountant teaching CA students. Use NFRS/NAS terminology.'
)
show('Without role', no_role)
show('With role',    with_role)

Without role
----------------------------------------------------------------------
Provisions and contingent liabilities are both accounting concepts that relate to potential future obligations, but they differ in terms of certainty and recognition in financial statements.

### Provisions:
1. **Definition**: Provisions are liabilities of uncertain timing or amount that a company recognizes in its financial statements. They are created when a company has a present obligation (legal or constructive) resulting from past events, and it is probable that an outflow of resources will be required to settle that obligation.

2. **Recognition**: Provisions are recognized in the financial statements when:
   - There is a present obligation.
   - It is probable that an outflow of resources will be required to settle the obligation.
   - The amount can be reliably estimated.

3. **Examples**: Common examples of provisions include:
   - Warranty obligations
   - Restructuring costs
   - Legal dispu

## Pattern 2 — Few-shot prompting

We show the model 2-3 examples of the format we want.

In [4]:
prompt = '''\
Classify each finding as HIGH / MEDIUM / LOW risk.

Finding: Vendor master has 2 entries for the same vendor.
Risk: MEDIUM

Finding: A purchase of NPR 8 lakh was made without dual approval.
Risk: HIGH

Finding: Bank reconciliation done 6 days after month-end (policy says 5).
Risk: LOW

Finding: Related-party purchase booked without benchmark pricing.
Risk:
'''
show('Few-shot', ask_llm(prompt))

Few-shot
----------------------------------------------------------------------
Finding: Related-party purchase booked without benchmark pricing.  
Risk: HIGH



## Pattern 3 — Structured output prompting

Ask for JSON so we can use the result downstream.

In [ ]:
from src.llm_client import ask_llm_json
memo = '''\
During the audit we noted three findings: (1) two purchase invoices exceeding NPR 5,00,000 
lacked dual approval, (2) the vendor master contains duplicate entries for Kathmandu Steel 
Suppliers, and (3) related-party purchases from Himal Family Enterprises lack benchmark pricing.
'''
result = ask_llm_json(
    f'Extract findings from the memo as a JSON list of {{"finding":..., "area":..., "risk":...}}. \n\nMemo:\n{memo}'
)
print(result)

## Pattern 4 — Chain-of-thought *output*

We ask the model to **show** its reasoning steps in the answer. This often improves accuracy.

In [ ]:
q = (
    'A company has revenue NPR 48.6 crore and net profit NPR 4.2 crore. '
    'Loan outstanding is NPR 18.2 crore at 10.5%. '
    'Roughly estimate the interest coverage ratio. Show working step by step.'
)
show('Reasoning output', ask_llm(q))

## Pattern 5 — Grounded prompting

Give the model the source. This is the foundation of RAG.

In [ ]:
policy = '''\
Procurement Policy, clause 1: For purchases above NPR 1,00,000 a minimum of three competitive quotations 
must be obtained. For purchases above NPR 10,00,000 a sealed-bid tender process is required. Sole-source 
procurement is permitted only with the CFO's prior written approval and documented justification.
'''
q = 'Per the policy below, can the company sole-source a NPR 12 lakh purchase? Quote the relevant clause.\n\n' + policy
show('Grounded answer', ask_llm(q))

## Hallucination demo (important!)

We deliberately ask about a fictional ICAN circular. Watch what the model invents.

In [ ]:
show('Possibly hallucinated', ask_llm('What does ICAN Technical Circular 99/2099-50 say about cryptoasset audit?'))

**Lesson.** The model may *describe a plausible-looking circular that does not exist*. Never rely on an LLM for citations of laws, standards, or circulars without an independent check.


## Practical CA prompts (try these live)

1. *Summarise this audit memo in 5 bullet points.*
2. *Extract risks and rate them HIGH/MEDIUM/LOW.*
3. *Draft a polite email asking a vendor to provide PAN documentation.*
4. *Explain a tax provision to a non-finance director.*
5. *Convert these messy notes into an audit checklist.*


In [ ]:
# Exercise — change the inputs below and observe the answers
your_memo = 'During inventory observation at Bhaktapur, slow-moving SKU FG-509 (NPR 2.16 lakh, 700+ days old) was noted.'
show('Summary', ask_llm(f'Summarise in 3 bullets: {your_memo}'))

## Reflection questions

* In which of your firm's workflows would few-shot prompting save the most time?
* What is the worst that could happen if a hallucinated tax citation reached a client?
* When is *structured-output* prompting better than free text for your work?


## Common errors

| Symptom | Fix |
|---|---|
| Model ignores your role | Move the role to the `system` argument, not the prompt body |
| JSON cell prints `_parse_error` | Re-run; if it persists, add `Respond with only JSON, no commentary.` |
| Mock answers everywhere | Set a key in `.env` and restart the kernel |


## ⚠️ Professional caution

* LLMs sound confident even when wrong — *especially* on standards, laws, and citations.
* Treat LLM output as a **draft from an articled assistant**, never as final professional work.
